# Synthesis and recommendation

Pulls every notebook's saved output together and states a position.

In [1]:

import sys, warnings
sys.path.insert(0, r"/Users/shaan/Desktop/FAM/sunpharma-organon-merger-arbitrage")
warnings.filterwarnings("ignore")

from src import config as cfg
import pandas as pd
pd.set_option("display.width", 160)

def load(name):
    return pd.read_csv(cfg.DATA_FINAL / f"{name}.csv", index_col=0)["value"]

deal = load("deal_snapshot")
valuation = load("valuation_summary")
risk = load("risk_summary")
events = load("event_study_summary")
ts = load("timeseries_summary")
arb = load("arb_portfolio_summary")
deriv = load("derivatives_summary")


## 1. Is the price fair?

In [2]:

print(f"Standalone DCF value/share, at a market-cross-validated discount rate : ${float(valuation['standalone_value_per_share']):.2f}")
print(f"  (WACC used {float(valuation['wacc_used']):.2%}, bracketed by an embedded-cost-of-debt floor of "
      f"{float(valuation['wacc_embedded']):.2%} and a synthetic-credit-rating ceiling of {float(valuation['wacc_marginal_synthetic']):.2%})")
print(f"Offer price                                                             : ${float(valuation['offer_price']):.2f}")
print(f"Revenue growth the offer requires (constant, 5yr)                        : {float(valuation['implied_revenue_growth_required']):.2%}")
print(f"  vs. actual trailing 2-year revenue CAGR                                : {float(valuation['historical_2yr_revenue_cagr']):.2%}")
print(f"Implied synergy / growth premium                                          : ${float(valuation['implied_synergy_value_usd_bn']):.2f}B "
      f"(no dollar figure was disclosed by management - this is this project's own estimate)")
print()
print(f"Acquirer (Sun Pharma) CAR around announcement (-5,+5)                     : {float(events['sun_car_announcement_pm5d']):+.2%}, "
      f"t={float(events['sun_car_announcement_tstat']):.2f} - positive and significant")


Standalone DCF value/share, at a market-cross-validated discount rate : $6.90
  (WACC used 6.61%, bracketed by an embedded-cost-of-debt floor of 5.29% and a synthetic-credit-rating ceiling of 7.90%)
Offer price                                                             : $14.00
Revenue growth the offer requires (constant, 5yr)                        : 3.87%
  vs. actual trailing 2-year revenue CAGR                                : -0.38%
Implied synergy / growth premium                                          : $1.86B (no dollar figure was disclosed by management - this is this project's own estimate)

Acquirer (Sun Pharma) CAR around announcement (-5,+5)                     : +9.62%, t=2.64 - positive and significant


In [3]:

print("Reading: the offer requires Organon's revenue to reverse from a two-year decline to roughly")
print("+3.9% annual growth - a real, non-trivial ask, but not an implausible one for a company that has")
print("just closed a divestiture and is being folded into a larger commercial platform. It is not a")
print("hockey-stick assumption. Combined with a positive, statistically significant market reaction in")
print("the ACQUIRER's own stock - the opposite of what a market judging the deal as overpaying would")
print("produce - the balance of evidence does not support a simple 'Sun Pharma overpaid' conclusion.")
print("Nor does it support 'obvious bargain': the required growth reversal is real, and the entire")
print("synergy estimate is this project's own construction, not a disclosed figure.")


Reading: the offer requires Organon's revenue to reverse from a two-year decline to roughly
+3.9% annual growth - a real, non-trivial ask, but not an implausible one for a company that has
just closed a divestiture and is being folded into a larger commercial platform. It is not a
hockey-stick assumption. Combined with a positive, statistically significant market reaction in
the ACQUIRER's own stock - the opposite of what a market judging the deal as overpaying would
produce - the balance of evidence does not support a simple 'Sun Pharma overpaid' conclusion.
Nor does it support 'obvious bargain': the required growth reversal is real, and the entire
synergy estimate is this project's own construction, not a disclosed figure.


## 2. What happened around the deal

In [4]:

print(f"Realised volatility: {float(risk['vol_baseline']):.1%} baseline -> {float(risk['vol_pre_leak']):.1%} pre-leak -> "
      f"{float(risk['vol_leak_to_announcement']):.1%} leak-to-announcement -> {float(risk['vol_post_announcement']):.1%} post-announcement")
print(f"  a ~{(1 - float(risk['vol_post_announcement'])/float(risk['vol_leak_to_announcement'])):.0%} collapse once the deal was signed - the stock is now priced")
print("  almost entirely off deal-completion risk, not business fundamentals.")
print()
print(f"Pre-announcement price drift: {float(events['pct_move_pre_announcement_simple']):.1%} of the total move (simple-return basis) happened")
print("  before the formal announcement. But the narrow-window CAR around the 16-Jan reference date is")
print("  NOT statistically significant, and there is no abnormal trading volume around it either")
print(f"  (volume ratio {float(events['leak_volume_ratio']):.2f}x, z={float(events['leak_volume_zscore']):.2f}). The evidence supports a large, gradual")
print("  re-rating over roughly three months, not a discrete, detectable leak event on a single day.")
print()
print(f"Cross-market causality: Organon's own past returns Granger-cause Sun Pharma's returns")
print(f"  (lag-1 p={float(ts['gc_ogn_to_sun_lag1_pvalue']):.4f}), but not the reverse (p={float(ts['gc_sun_to_ogn_lag1_pvalue']):.2f} lagged, "
      f"p={float(ts['sameday_sun_to_ogn_pvalue']):.2f} same-day) - consistent with Organon being the")
print("  deal-specific driver and Sun Pharma's own price mostly unaffected by it day to day.")
print()
print(f"No spillover to peers: {int(events['peer_spillover_significant_count'])} of 4 US pharma peers showed a significant CAR")
print("  around the announcement - the deal was idiosyncratic to these two names.")


Realised volatility: 44.8% baseline -> 58.0% pre-leak -> 94.2% leak-to-announcement -> 4.5% post-announcement
  a ~95% collapse once the deal was signed - the stock is now priced
  almost entirely off deal-completion risk, not business fundamentals.

Pre-announcement price drift: 69.6% of the total move (simple-return basis) happened
  before the formal announcement. But the narrow-window CAR around the 16-Jan reference date is
  NOT statistically significant, and there is no abnormal trading volume around it either
  (volume ratio 1.04x, z=0.05). The evidence supports a large, gradual
  re-rating over roughly three months, not a discrete, detectable leak event on a single day.

Cross-market causality: Organon's own past returns Granger-cause Sun Pharma's returns
  (lag-1 p=0.0001), but not the reverse (p=0.57 lagged, p=0.98 same-day) - consistent with Organon being the
  deal-specific driver and Sun Pharma's own price mostly unaffected by it day to day.

No spillover to peers: 0 of 4 

## 3. Should the spread be traded, at today's price?

In [5]:

print(f"Spot ${float(arb['spot']):.2f}, gross spread {float(arb['gross_spread']):.2%}, spread-implied P(completion) "
      f"{float(arb['implied_p_completion_unaffected_basis']):.1%}")
print(f"Breakeven P(completion) at the unaffected-price basis                    : {float(arb['breakeven_p_unaffected_basis']):.1%}")
print()
print(f"Forward Monte Carlo (assessed ~92% mean completion probability, not the market's ~94%):")
print(f"  mean holding-period return                                             : {float(arb['mc_mean_holding_return']):+.2%}")
print(f"  mean ANNUALISED return                                                  : below the {0.04617:.2%} risk-free rate")
print(f"  probability of a loss                                                    : {float(arb['mc_prob_loss']):.1%}")
print(f"  95% VaR / CVaR                                                            : {float(arb['mc_var95']):.1%} / {float(arb['mc_cvar95']):.1%}")
print(f"  max-Sharpe two-asset mix (arb position vs. S&P 500)                       : 0% arb / 100% index")
print()
print(f"Deal-break protection (protective put, K=$6.90, priced at the correct pre-leak vol, NOT the")
print(f"  circular deal-pinned vol)                                                 : ${float(deriv['put_premium_correct_vol']):.3f}/share")
print(f"  (a naive pricing at post-announcement vol would show ${float(deriv['put_premium_circular_vol']):.6f} - understating true cost by "
      f"{float(deriv['circular_pricing_understatement_pct']):.0%})")


Spot $13.57, gross spread 3.17%, spread-implied P(completion) 93.9%
Breakeven P(completion) at the unaffected-price basis                    : 93.9%

Forward Monte Carlo (assessed ~92% mean completion probability, not the market's ~94%):
  mean holding-period return                                             : -1.04%
  mean ANNUALISED return                                                  : below the 4.62% risk-free rate
  probability of a loss                                                    : 8.1%
  95% VaR / CVaR                                                            : -47.8% / -51.5%
  max-Sharpe two-asset mix (arb position vs. S&P 500)                       : 0% arb / 100% index

Deal-break protection (protective put, K=$6.90, priced at the correct pre-leak vol, NOT the
  circular deal-pinned vol)                                                 : $0.080/share
  (a naive pricing at post-announcement vol would show $0.000000 - understating true cost by 100%)


## Recommendation

In [6]:

print("PASS on the unhedged spread at today's price, conditional on the assumptions below - not a BUY,")
print("not a categorical avoid.")
print()
print("The trade only clears a reasonable risk/reward bar if the true completion probability is close to")
print(f"the market's own implied {float(arb['implied_p_completion_unaffected_basis']):.1%}, i.e. a break probability near "
      f"{1-float(arb['implied_p_completion_unaffected_basis']):.1%} rather than the more conservative ~8% used in the")
print("Monte Carlo above. Two facts support leaning toward the market's more optimistic number: shareholder")
print("approval was already obtained on 23-Jul-2026 - a major conditionality already cleared - and the")
print("acquirer's own stock reaction was positive, arguing against a deal the market expects to unravel.")
print()
print("Against that: this project did not verify what regulatory or other closing conditions remain")
print("outstanding as of the snapshot date. That is a real, unclosed gap, not a rounding error - anyone")
print("sizing this position for real capital should confirm the remaining closing-condition checklist")
print("before relying on either probability estimate here.")
print()
print("If a position is taken, a protective put struck near the unaffected price is the correctly-priced")
print("hedge available (${:.2f}/share at the pre-leak volatility regime) - not a naive one calibrated to".format(float(deriv['put_premium_correct_vol'])))
print("the deal-pinned volatility the market is currently quoting.")


PASS on the unhedged spread at today's price, conditional on the assumptions below - not a BUY,
not a categorical avoid.

The trade only clears a reasonable risk/reward bar if the true completion probability is close to
the market's own implied 93.9%, i.e. a break probability near 6.1% rather than the more conservative ~8% used in the
Monte Carlo above. Two facts support leaning toward the market's more optimistic number: shareholder
approval was already obtained on 23-Jul-2026 - a major conditionality already cleared - and the
acquirer's own stock reaction was positive, arguing against a deal the market expects to unravel.

Against that: this project did not verify what regulatory or other closing conditions remain
outstanding as of the snapshot date. That is a real, unclosed gap, not a rounding error - anyone
sizing this position for real capital should confirm the remaining closing-condition checklist
before relying on either probability estimate here.

If a position is taken, a pro

## Limitations

- The DCF's discount rate is calibrated to reproduce the unaffected market price, which validates internal consistency but is not an independent estimate of fair value - it is the market's own implied rate, cross-checked (not proven) against two independently modelled bounds.
- No dollar synergy figure was disclosed by Sun Pharma; every synergy number here is this project's own reverse-DCF construction.
- The break price used throughout ($6.90, the unaffected close, plus -10%/-20% sensitivity) is a proxy. A broken deal can print anywhere, including below the unaffected level if fundamentals have deteriorated in the interim.
- Outstanding regulatory/closing conditions as of the snapshot date were not verified against primary sources beyond the SEC filings reviewed.
- Sun Pharma's fiscal year (March) and Organon's (December) were reconciled to trailing-twelve-month figures where compared directly, but the two companies' "most recent annual" figures used elsewhere are not perfectly contemporaneous.
- All prices are as of the stated snapshot date; this is a point-in-time analysis, not a live-updating one.